# 6.2.5 LLM Evaluation

Compute BLEU-4, ROUGE-L, and Exact Match on synthetic reference/hypothesis pairs.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

Path('output').mkdir(exist_ok=True)

def ngrams(toks, n): return [tuple(toks[i:i+n]) for i in range(len(toks)-n+1)]

def bleu(ref, hyp, n=4):
    rt, ht = ref.lower().split(), hyp.lower().split()
    scores = []
    for k in range(1, n+1):
        ref_ng = Counter(ngrams(rt, k)); hyp_ng = Counter(ngrams(ht, k))
        clip = sum(min(c, ref_ng[g]) for g, c in hyp_ng.items())
        scores.append(clip / (sum(hyp_ng.values()) + 1e-10))
    bp = min(1.0, np.exp(1 - len(rt)/(len(ht)+1e-10)))
    return bp * np.exp(sum(0.25 * np.log(s+1e-10) for s in scores))

def rouge_l(ref, hyp):
    a, b = ref.lower().split(), hyp.lower().split()
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j] = dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j],dp[i][j-1])
    lcs = dp[m][n]; p = lcs/(n+1e-10); r = lcs/(m+1e-10)
    return 2*p*r/(p+r+1e-10)

pairs = [
    ('the cat sat on the mat', 'the cat sat on the mat'),
    ('the cat sat on the mat', 'a cat is sitting on a mat'),
    ('machine learning is powerful', 'machine learning is very powerful'),
    ('transformers use attention', 'attention is used in transformers'),
    ('neural networks learn features', 'deep nets extract representations'),
]
bleu_s = [bleu(r,h) for r,h in pairs]
rouge_s = [rouge_l(r,h) for r,h in pairs]
em_s = [float(r==h) for r,h in pairs]
for i,(b,ro,em) in enumerate(zip(bleu_s,rouge_s,em_s)):
    print(f'P{i+1}: BLEU={b:.3f}  ROUGE-L={ro:.3f}  EM={em:.0f}')

In [ ]:
x = np.arange(len(pairs)); w = 0.28
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x-w, bleu_s, w, label='BLEU-4', color='steelblue')
ax.bar(x,   rouge_s, w, label='ROUGE-L', color='darkorange')
ax.bar(x+w, em_s, w, label='EM', color='mediumseagreen')
ax.set(xticks=x, xticklabels=[f'P{i+1}' for i in range(len(pairs))],
       ylabel='Score', title='LLM Evaluation Metrics')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('output/llm_evaluation.png')
print('Saved llm_evaluation.png')